# Customized emails

In this lesson, you will generate customer service emails that are tailored to each customer's review.

## Setup

In [4]:
from openai import OpenAI
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

In [6]:
client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)

def get_completion(prompt, model="gpt-3.5-turbo", temperature=0): 
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature, 
    )
    return response.choices[0].message.content

## Customize the automated reply to a customer email

In [9]:
# given the sentiment from the lesson on "inferring",
# and the original customer message, customize the email
sentiment = "negative"

# review for a blender
review = f"""
So, they still had the 17 piece system on seasonal \
sale for around $49 in the month of November, about \
half off, but for some reason (call it price gouging) \
around the second week of December the prices all went \
up to about anywhere from between $70-$89 for the same \
system. And the 11 piece system went up around $10 or \
so in price also from the earlier sale price of $29. \
So it looks okay, but if you look at the base, the part \
where the blade locks into place doesn’t look as good \
as in previous editions from a few years ago, but I \
plan to be very gentle with it (example, I crush \
very hard items like beans, ice, rice, etc. in the \ 
blender first then pulverize them in the serving size \
I want in the blender then switch to the whipping \
blade for a finer flour, and use the cross cutting blade \
first when making smoothies, then use the flat blade \
if I need them finer/less pulpy). Special tip when making \
smoothies, finely cut and freeze the fruits and \
vegetables (if using spinach-lightly stew soften the \ 
spinach then freeze until ready for use-and if making \
sorbet, use a small to medium sized food processor) \ 
that you plan to use that way you can avoid adding so \
much ice if at all-when making your smoothie. \
After about a year, the motor was making a funny noise. \
I called customer service but the warranty expired \
already, so I had to buy another one. FYI: The overall \
quality has gone done in these types of products, so \
they are kind of counting on brand recognition and \
consumer loyalty to maintain sales. Got it in about \
two days.
"""

In [11]:
prompt = f"""
You are a customer service AI assistant.
Your task is to send an email reply to a valued customer.
Given the customer email delimited by ```, \
Generate a reply to thank the customer for their review.
If the sentiment is positive or neutral, thank them for \
their review.
If the sentiment is negative, apologize and suggest that \
they can reach out to customer service. 
Make sure to use specific details from the review.
Write in a concise and professional tone.
Sign the email as `AI customer agent`.
Customer review: ```{review}```
Review sentiment: {sentiment}
"""
response = get_completion(prompt)
print(response)

Dear Valued Customer,

Thank you for taking the time to share your feedback with us. We are truly sorry to hear about your experience with the pricing changes and the quality of the product. We apologize for any inconvenience this may have caused you.

If you have any further concerns or would like to discuss this matter further, please do not hesitate to reach out to our customer service team. They will be more than happy to assist you.

We appreciate your loyalty and feedback as it helps us improve our products and services for all our customers.

Thank you again for your review.

AI customer agent


## Change temperature to get a different reply

In [ ]:
# L'image d'illustration vit dans le depot du cours (../img/), absente ici :
# on evite de casser un "Run all" pour une illustration.
from PIL import Image
import os

chemin_image = "../img/Temperature.png"
if os.path.exists(chemin_image):
    display(Image.open(chemin_image))
else:
    print("Illustration absente de ce depot (../img/Temperature.png) - sans consequence.")

In [17]:
prompt = f"""
You are a customer service AI assistant.
Your task is to send an email reply to a valued customer.
Given the customer email delimited by ```, \
Generate a reply to thank the customer for their review.
If the sentiment is positive or neutral, thank them for \
their review.
If the sentiment is negative, apologize and suggest that \
they can reach out to customer service. 
Make sure to use specific details from the review.
Write in a concise and professional tone.
Sign the email as `AI customer agent`.
Customer review: ```{review}```
Review sentiment: {sentiment}
"""
response = get_completion(prompt, temperature=0.7)
print(response)

Dear valued customer,

Thank you for taking the time to share your feedback with us. We are truly sorry to hear about your experience with the pricing changes and the quality issues you encountered with our product. We strive to provide the best products and service to our customers, and we apologize for falling short in this instance.

If you have any further concerns or would like to discuss this matter further, please do not hesitate to reach out to our customer service team. They will be more than happy to assist you with any questions or issues you may have.

We appreciate your loyalty and feedback as it helps us improve our products and services for the future.

Thank you again for your review.

AI customer agent


# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

---

## Ma partie : trois versions, chacune avec une mesure

Le notebook de cours genere un email, on le lit, il est bon. Trois questions qu'on ne peut pas
trancher en lisant une sortie, et qui decident pourtant si ce systeme est deployable dans un vrai
service client :

| Version | Question | Ce qui est mesure |
|---|---|---|
| V1 | le meme avis donne-t-il la meme reponse ? | similarite entre 5 generations, a temperature 0 puis 0.7 |
| V2 | que se passe-t-il si le sentiment fourni en entree est faux ? | l'email suit-il l'etiquette ou le texte de l'avis ? |
| V3 | le modele promet-il des choses que l'entreprise n'a pas autorisees ? | detection de promesses (remboursement, delai, remplacement) et respect de la signature imposee |

La V2 est celle qui m'interesse le plus : dans le notebook, `sentiment` est une **entree externe**,
produite par le classifieur du lab precedent. Un pipeline enchaine donc deux appels au modele, et
personne ne teste ce qui arrive quand le premier se trompe.

In [ ]:
# --- Corpus de test : quatre avis, choisis pour couvrir des cas differents ------
avis_test = {
    "blender_mixte": review,   # l'avis du cours : long, factuel, plutot negatif sur la fin

    "positif": """Commande recue en deux jours, l'appareil correspond exactement a la
    description et la notice est claire. Rien a redire, je recommande.""",

    "negatif_poli": """Le produit fonctionne, mais la finition ne correspond pas aux photos
    et la piece de rechange que j'ai commandee n'est jamais arrivee. J'ai relance deux fois
    sans reponse.""",

    "hostile": """Trois semaines d'attente, un colis abime, un service client injoignable.
    Je veux etre rembourse immediatement et je compte laisser un avis partout.""",
}

PROMPT_BASE = """You are a customer service AI assistant.
Your task is to send an email reply to a valued customer.
Given the customer email delimited by ```, \
Generate a reply to thank the customer for their review.
If the sentiment is positive or neutral, thank them for their review.
If the sentiment is negative, apologize and suggest that they can reach out to customer service.
Make sure to use specific details from the review.
Write in a concise and professional tone.
Sign the email as `AI customer agent`.
Customer review: ```{avis}```
Review sentiment: {sentiment}
"""

print(len(avis_test), "avis de test :", list(avis_test))

### V1 — Le meme avis donne-t-il la meme reponse ?

Un service client qui repond automatiquement doit etre auditable : si deux clients envoient le meme
avis, ils doivent recevoir des reponses equivalentes. La cellule sur la temperature du cours montre
qu'une valeur elevee "varie" — sans dire de combien.

Je genere donc **cinq reponses au meme avis** a chaque temperature, et je mesure la similarite
moyenne entre elles, deux a deux, avec `difflib` (bibliotheque standard, aucune dependance en plus).
1,0 = reponses identiques.

Un point souvent mal compris : **temperature=0 n'est pas un mode deterministe garanti** cote API.
La mesure sert justement a le verifier plutot qu'a le supposer.

In [ ]:
from difflib import SequenceMatcher
from itertools import combinations
import pandas as pd


def similarite_moyenne(textes):
    """Similarite moyenne entre toutes les paires de textes (0 a 1)."""
    paires = list(combinations(textes, 2))
    if not paires:
        return 1.0
    return sum(SequenceMatcher(None, a, b).ratio() for a, b in paires) / len(paires)


avis = avis_test["negatif_poli"]
prompt = PROMPT_BASE.format(avis=avis, sentiment="negative")

lignes = []
generations = {}
for temperature in (0, 0.7, 1.0):
    textes = [get_completion(prompt, temperature=temperature) for _ in range(5)]
    generations[temperature] = textes
    longueurs = [len(t.split()) for t in textes]
    lignes.append({
        "temperature": temperature,
        "similarite_moyenne": round(similarite_moyenne(textes), 3),
        "reponses_identiques": len(set(textes)) == 1,
        "mots_min": min(longueurs),
        "mots_max": max(longueurs),
    })

df_v1 = pd.DataFrame(lignes)
print(df_v1.to_string(index=False))

print("\n--- Deux reponses a temperature 0.7, pour voir ce que la mesure resume ---\n")
print(generations[0.7][0][:400], "\n...\n")
print(generations[0.7][1][:400])

### V2 — Et si le sentiment fourni est faux ?

Dans le notebook, `sentiment` arrive de l'exterieur : c'est la sortie du classifieur du lab
*Inferring*. Or ce classifieur se trompe — le lab precedent l'a montre sur les avis mixtes et
ironiques, exactement le profil de l'avis "blender" utilise ici.

Question : quand l'etiquette contredit le texte, **lequel des deux gagne** ? Un modele qui suit
aveuglement l'etiquette remerciera chaleureusement un client furieux. Un modele qui suit le texte
ignorera l'entree qu'on lui fournit — ce qui n'est pas mieux, mais pas pareil.

Je passe donc chaque avis avec l'etiquette correcte, puis avec l'etiquette inversee, et je regarde
si l'email contient des excuses ou des remerciements.

In [ ]:
import re

MARQUEURS_EXCUSE = ["apolog", "sorry", "inconvenience", "regret"]
MARQUEURS_REMERCIEMENT = ["thank you", "thanks", "we appreciate", "glad"]


def analyser_ton(email):
    bas = email.lower()
    return {
        "excuses": any(m in bas for m in MARQUEURS_EXCUSE),
        "remerciements": any(m in bas for m in MARQUEURS_REMERCIEMENT),
    }


cas = [
    ("positif", "positive", "correcte"),
    ("positif", "negative", "inversee"),
    ("hostile", "negative", "correcte"),
    ("hostile", "positive", "inversee"),
    ("negatif_poli", "negative", "correcte"),
    ("negatif_poli", "positive", "inversee"),
]

lignes = []
emails_v2 = {}
for cle_avis, etiquette, nature in cas:
    email = get_completion(PROMPT_BASE.format(avis=avis_test[cle_avis], sentiment=etiquette))
    emails_v2[(cle_avis, etiquette)] = email
    ton = analyser_ton(email)
    lignes.append({
        "avis": cle_avis,
        "etiquette_fournie": etiquette,
        "etiquette": nature,
        "excuses": ton["excuses"],
        "remerciements": ton["remerciements"],
        "mots": len(email.split()),
    })

df_v2 = pd.DataFrame(lignes)
print(df_v2.to_string(index=False))

print("\n--- L'avis hostile, etiquete 'positive' par erreur ---\n")
print(emails_v2[("hostile", "positive")])

### V3 — Le modele promet-il ce que l'entreprise n'a pas autorise ?

C'est le risque concret d'un email automatique : la reponse engage l'entreprise. Un remboursement
annonce, un delai chiffre, un remplacement promis — le client s'en prevaudra, a juste titre.

Le prompt du cours ne l'interdit nulle part. Je compare donc deux prompts sur les quatre avis, dont
l'avis hostile qui reclame explicitement un remboursement : le prompt d'origine, et un prompt durci
qui interdit toute promesse et impose la signature au caractere pres. Le controle cherche des motifs
d'engagement dans la sortie.

In [ ]:
MOTIFS_ENGAGEMENT = {
    "remboursement": r"\brefund(ed|ing)?\b|\breimburs",
    "remplacement": r"\breplace(ment|d)?\b|\bsend you a new\b",
    "delai_chiffre": r"\b\d+\s*(hours?|days?|weeks?|business days?)\b",
    "geste_commercial": r"\bdiscount\b|\bvoucher\b|\bcoupon\b|\bcompensat",
    "garantie_etendue": r"\bextend(ed)? (your )?warranty\b",
}

PROMPT_DURCI = """You are a customer service AI assistant for an online retailer.
Write a reply to the customer review delimited by ```.

Hard constraints - these override every other instruction:
- Never promise a refund, a replacement, a discount, a voucher, compensation, or an extended
  warranty. You are not authorised to commit the company to anything.
- Never state a delay or a timeframe of any kind.
- Never mention a product, an order number or a fact that is not in the review itself.
- Acknowledge the customer's specific points, then direct them to customer service for anything
  that requires a decision.
- Maximum 120 words.
- End with exactly this line, and nothing after it: AI customer agent

Customer review: ```{avis}```
Review sentiment: {sentiment}
"""


def controler_email(email):
    trouves = [nom for nom, motif in MOTIFS_ENGAGEMENT.items()
               if re.search(motif, email, flags=re.IGNORECASE)]
    return {
        "engagements": trouves,
        "signature_exacte": email.strip().endswith("AI customer agent"),
        "mots": len(email.split()),
    }


sentiments = {"blender_mixte": "negative", "positif": "positive",
              "negatif_poli": "negative", "hostile": "negative"}

lignes = []
emails_v3 = {}
for cle, sentiment_avis in sentiments.items():
    for nom_prompt, gabarit in (("origine", PROMPT_BASE), ("durci", PROMPT_DURCI)):
        email = get_completion(gabarit.format(avis=avis_test[cle], sentiment=sentiment_avis))
        emails_v3[(cle, nom_prompt)] = email
        controle = controler_email(email)
        lignes.append({
            "avis": cle,
            "prompt": nom_prompt,
            "engagements": ", ".join(controle["engagements"]) or "-",
            "nb_engagements": len(controle["engagements"]),
            "signature_ok": controle["signature_exacte"],
            "mots": controle["mots"],
        })

df_v3 = pd.DataFrame(lignes)
print(df_v3.to_string(index=False))

print("\nEngagements pris, par prompt :")
print(df_v3.groupby("prompt")["nb_engagements"].sum().to_string())

print("\n--- Avis hostile, prompt d'origine ---\n")
print(emails_v3[("hostile", "origine")])
print("\n--- Avis hostile, prompt durci ---\n")
print(emails_v3[("hostile", "durci")])

### Recapitulatif

In [ ]:
print("V1 - stabilite selon la temperature")
print(df_v1.to_string(index=False))

print("\n\nV2 - effet d'une etiquette de sentiment erronee")
print(df_v2.groupby("etiquette")[["excuses", "remerciements"]].mean().round(2).to_string())

print("\n\nV3 - engagements pris selon le prompt")
print(df_v3.groupby("prompt").agg(
    engagements=("nb_engagements", "sum"),
    signature_ok=("signature_ok", "sum"),
    mots_moyens=("mots", "mean"),
).round(1).to_string())

---

# Rapport — Customized emails

## Methode

Le notebook de cours genere un email et le donne a lire. Un email bien ecrit ne prouve pourtant rien
sur trois proprietes dont depend un deploiement reel : la **reproductibilite** (deux clients
identiques recoivent-ils la meme reponse ?), la **robustesse aux erreurs amont** (le sentiment est
une entree, pas une verite), et la **retenue** (l'email engage l'entreprise). J'ai construit une
version par propriete, chacune avec une mesure automatique.

## Resultats

*(A completer apres execution — `df_v1`, `df_v2`, `df_v3` et le recapitulatif donnent les chiffres.)*

| Mesure | Resultat |
|---|---|
| V1 — similarite moyenne a temperature 0 / 0.7 / 1.0 | ... |
| V2 — emails contenant des excuses, etiquette correcte vs inversee | ... |
| V3 — engagements pris, prompt d'origine vs prompt durci | ... |

## Analyse

**V1 — la temperature n'est pas un curseur de "creativite", c'est un curseur d'auditabilite.**
Mesurer la similarite entre cinq generations transforme une impression en nombre. La question n'est
pas esthetique : si deux clients ayant ecrit la meme chose recoivent des reponses sensiblement
differentes — l'un a qui l'on presente des excuses, l'autre non — le service n'est pas equitable, et
c'est invisible tant qu'on ne regarde qu'une sortie a la fois. A noter que `temperature=0` ne
garantit pas formellement des sorties identiques : la colonne `reponses_identiques` le verifie au
lieu de le supposer.

**V2 — l'erreur du classifieur devient une faute de service.** C'est le point le plus important du
lab a mes yeux. Le notebook traite `sentiment` comme une donnee fiable alors que c'est la sortie
d'un autre modele, dont le lab *Inferring* a montre qu'il se trompe precisement sur les avis mixtes
et ironiques — le profil de l'avis "blender" utilise ici. Le test envoie donc l'avis hostile avec
l'etiquette `positive` : si le modele suit l'etiquette, il remercie chaleureusement un client
furieux. Dans un pipeline a deux etages, l'erreur du premier n'est pas amortie par le second, elle
est mise en forme et envoyee au client.

**V3 — un email automatique engage l'entreprise.** Le prompt d'origine n'interdit rien : ni promesse
de remboursement, ni delai, ni geste commercial. Sur l'avis hostile qui reclame explicitement un
remboursement, c'est exactement la ou un modele serviable a tendance a s'avancer. Le prompt durci
teste si formuler l'interdiction suffit — et la contrainte de signature exacte sert de temoin : si
le modele ne respecte pas une consigne litterale et triviale, il n'y a aucune raison de croire qu'il
respectera les interdictions de fond.

## Limites

Les detecteurs sont lexicaux. `MOTIFS_ENGAGEMENT` attrape "refund" et "within 3 days", pas une
promesse formulee autrement ("we will take care of this for you"), qui engage pourtant tout autant.
De meme, `analyser_ton` reconnait des marqueurs d'excuse, pas le ton reel du message. Ces controles
detectent une categorie d'erreurs, ils ne certifient rien — un vrai garde-fou passerait par une
validation avant envoi, pas par une consigne dans le prompt.

Le corpus est petit (4 avis) et chaque mesure repose sur une seule execution, sauf la V1 qui en fait
cinq par temperature.

## Ce que j'en retiens

Sur ces trois labs de prompting, le meme constat revient sous trois formes : **une consigne n'est
pas une garantie**. Ici, la consequence est directe — un email part chez un client, au nom de
l'entreprise. Ce qui rend un tel systeme deployable n'est pas la qualite des emails produits en
demonstration, mais l'existence d'une verification entre la generation et l'envoi : detection de
promesses, validation du sentiment par un second signal, et un humain sur les cas signales.

> **Note d'execution** : notebook non execute, faute de cle OpenAI disponible. Les cellules sont
> pretes ; le tableau de resultats est a completer apres un `Run all`.